# 07 — Evaluation and probability calibration

The sigmoid mapping was fitted to out-of-fold scores from the training-only expanding folds. Validation labels assessed the mapping; they did not fit it. The final chronological test was evaluated only after model and threshold selection were frozen. Average Precision and trapezoidal area under the precision-recall curve are reported separately.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
results = json.loads((ROOT / "reports/ml/latest_results.json").read_text(encoding="utf-8"))
calibration = results["calibration"]
summary = pd.DataFrame.from_dict(
    {
        name: {
            "Brier score": value["brier_score"],
            "Log loss": value["log_loss"],
            "Equal-frequency ECE": value["expected_calibration_error"],
        }
        for name, value in (
            ("Raw score", calibration["raw"]),
            ("Training-fold sigmoid", calibration["sigmoid"]),
        )
    },
    orient="index",
)
display(summary)
display(
    pd.DataFrame(
        {
            "Raw score": calibration["raw"]["probability_quantiles"],
            "Sigmoid": calibration["sigmoid"]["probability_quantiles"],
        }
    )
)
print("Validation fraud count:", calibration["evaluation_fraud_count"])
print("Isotonic:", calibration["isotonic_status"])
display(Image(filename=str(ROOT / "reports/ml/figures/calibration_reliability.png")))
display(Image(filename=str(ROOT / "reports/ml/figures/probability_distributions.png")))